In [ ]:
!pip install -q ultralytics roboflow onnx onnxruntime

In [ ]:
from roboflow import Roboflow

API_KEY = "Roboflow api key"

rf = Roboflow(api_key=API_KEY)

project = rf.workspace(
    "indian-institute-of-technology-madras-xamot"
).project("pothole-detection-huf2x")

dataset = project.version(1).download("yolov8")

print("Dataset downloaded:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Pothole-detection-1 in yolov8:: 100%|██████████| 5456/5456 [00:01<00:00, 2791.50it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Dataset downloaded: /content/Pothole-detection-1


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=10,
    project="Pothole_Project",
    name="YOLOv8_Training"
)


Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Pothole-detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLOv8_Training, nbs=64, nms=None, opset=None, op

In [ ]:
best_model = YOLO(
    "/content/runs/detect/Pothole_Project/YOLOv8_Training/weights/best.pt"
)

metrics = best_model.val(
    data=f"{dataset.location}/data.yaml",
    split="test",
    imgsz=640
)

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,193 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1190.4±643.0 MB/s, size: 63.3 KB)
val: Scanning /content/Pothole-detection-1/test/labels... 274 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 274/274 368.8it/s 0.7s
val: New cache created: /content/Pothole-detection-1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.9it/s 4.7s
                   all        274        861      0.392      0.394      0.362      0.164
       crocodile crack         39         74      0.363      0.284      0.275     0.0959
    longitudinal crack         52        110      0.244      0.245       0.17     0.0629
               pothole        270        677       0.57      0.653      0.642      0.334
Speed: 1.6ms preprocess, 4.6ms inference, 0.0ms loss, 2.0ms p

In [ ]:
best_model.export(
    format="onnx",
    imgsz=640,
    opset=12,
    simplify=True,
    dynamic=False
)

print("ONNX export completed!")

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 73 layers, 3,133,193 parameters, 0 gradients, 8.6 GFLOPs

PyTorch: starting from '/content/runs/detect/Pothole_Project/YOLOv8_Training/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (6.2 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 10 packages in 238ms
Prepared 2 packages in 30ms
Installed 2 packages in 5ms
 + colorama==0.4.6
 + onnxslim==0.1.99

requirements: AutoUpdate success ✅ 0.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 12...
ONNX: slimming with onnxslim 0.1.99...
ONNX: export success ✅ 1.9s, saved a

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(
    "/content/runs/detect/Pothole_Project/YOLOv8_Training/weights/best.onnx",
    providers=["CPUExecutionProvider"]
)

print("ONNX model loaded successfully")
print("Input:", session.get_inputs()[0].shape)
print("Output:", session.get_outputs()[0].shape)

ONNX model loaded successfully
Input: [1, 3, 640, 640]
Output: [1, 7, 8400]
